# 09 — Transfer Learning

Reuse a model pretrained on a large dataset (e.g. ImageNet) instead of training from scratch — faster convergence, better performance on small datasets.

## Loading a Pretrained Model

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model


## Inspecting & Replacing the Head

In [ ]:
print(model.fc)   # Linear(in_features=512, out_features=1000)

num_classes = 10
model.fc = nn.Linear(model.fc.in_features, num_classes)
print(model.fc)   # Linear(in_features=512, out_features=10)


## Strategy 1: Feature Extraction (freeze backbone)

Freeze all pretrained weights; only train the new head. Best when your dataset is small and similar to the pretraining domain.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)   # new layer has requires_grad=True by default

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)   # only optimize the head


## Strategy 2: Fine-Tuning (unfreeze everything)

Train the whole network with a small learning rate. Best when you have more data and it differs from the pretraining domain.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = True   # already default, shown for clarity

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)   # smaller LR than training from scratch


## Strategy 3: Partial Unfreeze (unfreeze last block only)

Compromise: keep early (generic) layers frozen, fine-tune later (task-specific) layers.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():   # unfreeze last residual block
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, num_classes)   # new head, trainable by default

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=1e-4)


## Strategy 4: Discriminative Learning Rates

Use smaller LR for early (generic) layers, larger LR for later (task-specific) layers and the new head.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)

optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 5e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(),     "lr": 1e-3},
])


## Full Transfer Learning Training Loop

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ImageNet normalization stats — required since the backbone was pretrained on ImageNet
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

train_ds = datasets.ImageFolder("data/train", transform=train_transform)
val_ds = datasets.ImageFolder("data/val", transform=val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

best_acc = 0.0
for epoch in range(10):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(1)
            correct += (preds == yb).sum().item()
    val_acc = correct / len(val_ds)
    print(f"epoch {epoch+1}: val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")


## Two-Phase Recipe (common in practice)

1. Freeze backbone, train only the head for a few epochs until it stabilizes.
2. Unfreeze some/all layers, drop the LR by 10x, continue training (fine-tuning).

In [ ]:
# Phase 1: warm up the head
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, num_classes)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
# ... train a few epochs ...

# Phase 2: unfreeze and fine-tune at lower LR
for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
# ... continue training ...


## Other Pretrained Backbones

In [ ]:
models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)     # Vision Transformer
models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)  # lightweight, mobile-friendly


## Common Gotchas
- Forgetting to replace the classification head's output size (`model.fc`) leaves it at 1000 ImageNet classes.
- Must use the same normalization stats the backbone was pretrained with (check `weights.transforms()` for the recommended preprocessing).
- `model.eval()` matters even more here — many pretrained backbones use BatchNorm, and forgetting eval mode during inference uses batch statistics instead of running stats.
- Freezing params via `requires_grad = False` does NOT exclude them from `optimizer.param_groups` automatically — must pass only the trainable params (or filter them) when constructing the optimizer.